In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score, classification_report
)
from sklearn.pipeline import Pipeline
import warnings
warnings.filterwarnings('ignore')

# === 1. Загрузка и первичный осмотр данных ===
df = pd.read_csv('cardio_train.csv', sep=';')

# Преобразуем возраст из дней в годы
df['age_years'] = (df['age'] / 365.25).astype(int)

# Удалим исходный 'age' и 'id'
df = df.drop(columns=['id', 'age'])

# === 2. Очистка данных ===
# Удалим очевидные выбросы:
# - рост < 120 см или > 220 см
# - вес < 30 кг или > 200 кг
# - диастолическое давление > систолического
# - давление < 60 или > 220
df = df[
    (df['height'] >= 120) & (df['height'] <= 220) &
    (df['weight'] >= 30) & (df['weight'] <= 200) &
    (df['ap_hi'] >= 60) & (df['ap_hi'] <= 220) &
    (df['ap_lo'] >= 40) & (df['ap_lo'] <= 150) &
    (df['ap_hi'] > df['ap_lo'])
]

# Вычислим BMI
df['bmi'] = df['weight'] / ((df['height'] / 100) ** 2)

# Удалим исходные height/weight для упрощения
df = df.drop(columns=['height', 'weight'])

# Разделим признаки и целевую переменную
X = df.drop(columns=['cardio'])
y = df['cardio']

# === 3. Настройка кросс-валидации ===
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# === 4. Определение моделей ===
models = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(random_state=42, max_iter=1000))
    ]),
    'RandomForest': RandomForestClassifier(
        n_estimators=100,
        random_state=42,
        class_weight='balanced'  # компенсация дисбаланса
    )
}

# === 5. Обучение и оценка ===
results = {}

for name, model in models.items():
    print(f"\n=== Оценка модели: {name} ===")
    scores = {'roc_auc': [], 'f1': [], 'precision': [], 'recall': []}
    
    for fold, (train_idx, val_idx) in enumerate(cv.split(X, y)):
        X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
        y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]
        
        model.fit(X_train, y_train)
        y_pred = model.predict(X_val)
        y_proba = model.predict_proba(X_val)[:, 1]
        
        scores['roc_auc'].append(roc_auc_score(y_val, y_proba))
        scores['f1'].append(f1_score(y_val, y_pred))
        scores['precision'].append(precision_score(y_val, y_pred))
        scores['recall'].append(recall_score(y_val, y_pred))
    
    # Усреднение по фолдам
    avg_scores = {metric: np.mean(vals) for metric, vals in scores.items()}
    results[name] = avg_scores
    
    print(f"ROC-AUC: {avg_scores['roc_auc']:.4f}")
    print(f"F1-score: {avg_scores['f1']:.4f}")
    print(f"Precision: {avg_scores['precision']:.4f}")
    print(f"Recall: {avg_scores['recall']:.4f}")

# === 6. Вывод лучших результатов ===
print("\n=== ИТОГОВЫЕ РЕЗУЛЬТАТЫ ===")
for name, scores in results.items():
    print(f"\n{name}:")
    for metric, value in scores.items():
        print(f"  {metric.upper()}: {value:.4f}")

# === 7. Дополнительно: полный отчёт по лучшей модели ===
best_model_name = max(results.keys(), key=lambda k: results[k]['roc_auc'])
best_model = models[best_model_name]

# Обучим на всём датасете для финального отчёта
best_model.fit(X, y)
y_full_pred = best_model.predict(X)
print(f"\n=== Классификационный отчёт для {best_model_name} (на всём датасете) ===")
print(classification_report(y, y_full_pred))


=== Оценка модели: LogisticRegression ===
ROC-AUC: 0.7911
F1-score: 0.7076
Precision: 0.7540
Recall: 0.6666

=== Оценка модели: RandomForest ===
ROC-AUC: 0.7454
F1-score: 0.6862
Precision: 0.6826
Recall: 0.6898

=== ИТОГОВЫЕ РЕЗУЛЬТАТЫ ===

LogisticRegression:
  ROC_AUC: 0.7911
  F1: 0.7076
  PRECISION: 0.7540
  RECALL: 0.6666

RandomForest:
  ROC_AUC: 0.7454
  F1: 0.6862
  PRECISION: 0.6826
  RECALL: 0.6898

=== Классификационный отчёт для LogisticRegression (на всём датасете) ===
              precision    recall  f1-score   support

           0       0.71      0.79      0.74     34669
           1       0.75      0.67      0.71     33934

    accuracy                           0.73     68603
   macro avg       0.73      0.73      0.73     68603
weighted avg       0.73      0.73      0.73     68603

